In [1]:
!pip install -q pyannote.audio pyannote.metrics

import os, shutil, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.

In [2]:
!ls /kaggle/input/          # confirm both slugs

datasets  notebooks


In [3]:
import pathlib, shutil, os

ROOT = pathlib.Path("/kaggle/input")

# Find the directory that actually contains the scripts, wherever it landed.
CODE = next(p.parent for p in ROOT.rglob("stage3_diarize.py"))
# Find the directory that actually contains the WAVs.
AUDIO = next(p.parent for p in ROOT.rglob("*.wav"))
REF = next(p.parent for p in ROOT.rglob("clip_meta.csv"))
WORK = "/kaggle/working/data"

print("CODE  =", CODE)
print("AUDIO =", AUDIO, f"({len(list(AUDIO.glob('*.wav')))} wavs)")
print("REF   =", REF, f"({len(list(REF.rglob('*')))} files)")

CODE  = /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code
AUDIO = /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav (99 wavs)
REF   = /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code/ref (205 files)


In [4]:
pathlib.Path(WORK).mkdir(parents=True, exist_ok=True)
shutil.copytree(REF, f"{WORK}/ref", dirs_exist_ok=True)
for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")
mf = next(ROOT.rglob("manifest.jsonl"), None)
if mf: shutil.copy(mf, WORK)

print("ref files:", len(list(pathlib.Path(f'{WORK}/ref').rglob('*'))))
print("scripts  :", [p.name for p in pathlib.Path('/kaggle/working').glob('*.py')])

ref files: 205
scripts  : ['stage2_parse_refs.py', 'stage1_extract.py', 'build_notebooks.py', 'stage3_score.py', 'stage3_diarize.py']


In [5]:
import shutil, pathlib
PREV = pathlib.Path("/kaggle/input/notebooks/ritankarmondal/sarvam-initial")   # fix the slug
DST  = pathlib.Path("/kaggle/working/data")

assert (PREV / "data").exists(), sorted(p.name for p in PREV.iterdir())
DST.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV / "data", DST, dirs_exist_ok=True)

print("ref  :", len(list((DST / "ref/rttm").glob("*.rttm"))))
print("hyp  :", len(list((DST / "hyp").rglob("*.rttm"))))

ref  : 100
hyp  : 173


In [6]:
!pip install -q "nemo_toolkit[asr]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.8/242.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

In [7]:
# !python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO}

In [8]:
# !python stage3_diarize.py --system community1 --data data --wav-dir {AUDIO} --limit 5
# !python stage3_diarize.py --system community1 --data data --wav-dir {AUDIO}

In [9]:
import shutil, pathlib
PREV = pathlib.Path("/kaggle/input/notebooks/ritankarmondal/sarvam-initial")   # fix the slug
DST  = pathlib.Path("/kaggle/working/data")

assert (PREV / "data").exists(), sorted(p.name for p in PREV.iterdir())
DST.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV / "data", DST, dirs_exist_ok=True)

print("ref  :", len(list((DST / "ref/rttm").glob("*.rttm"))))
print("hyp  :", len(list((DST / "hyp").rglob("*.rttm"))))

ref  : 100
hyp  : 173


In [10]:
!python stage3_score.py --data data --systems pyannote31 sortformer

[ok] scored pyannote31: 99 clips
[ok] scored sortformer: 99 clips

STAGE 3 -- BASELINE DIARIZATION   (collar=0.0, skip_overlap=False, UEM=full clip)
system             DER    miss      FA    conf     JER  spk acc  spk MAE
------------------------------------------------------------------------------
pyannote31      27.34%  11.59%   5.89%   9.86%  38.14%    72.7%     0.33
sortformer      74.85%  65.14%   2.19%   7.52%  65.29%    43.4%     1.55
------------------------------------------------------------------------------
DER/miss/FA/conf are duration-weighted. Macro (per-clip mean) for contrast:
  pyannote31     DER_macro  29.76%   JER_macro  38.09%   (weighted DER 27.34%)
  sortformer     DER_macro  52.02%   JER_macro  59.57%   (weighted DER 74.85%)
    [!] 25 clip(s) had NO hypothesis (scored as total miss)

------------------------------------------------------------------------------
DER by reference speaker count (duration-weighted within each bucket):
-----------------------------

In [11]:
import pandas as pd
pd.read_csv("/kaggle/working/data/results/diarization_summary.csv")

,system,n_clips,n_hyp_missing,DER,miss,false_alarm,confusion,DER_pyannote_accum,JER_pyannote_accum,DER_macro,JER_macro,spk_count_acc,spk_count_mae,spk_count_bias
0,pyannote31,99,0,0.273380,0.115851,0.058910,0.098620,0.273380,0.381357,0.29759,0.380920,0.727273,0.333333,-0.131313
1,sortformer,99,25,0.748509,0.651379,0.021889,0.075241,0.748509,0.652872,0.52021,0.595657,0.434343,1.545455,-1.383838


In [12]:
import collections
import json
import pathlib

data_dir = pathlib.Path("/kaggle/working/data")
manifest = data_dir / "manifest.jsonl"
hyp_dir = data_dir / "hyp"

rttm_count = len(list(hyp_dir.rglob("*.rttm"))) if hyp_dir.exists() else 0

print(f"Data directory:  {data_dir.exists()}")
print(f"Manifest exists: {manifest.exists()}")
print(f"Hyp RTTM count:  {rttm_count}")

if not manifest.exists():
    print("\n[!] manifest.jsonl not found in /kaggle/working/data/")
else:
    recs = [json.loads(line) for line in manifest.open(encoding="utf-8") if line.strip()]
    status_counts = collections.Counter(r.get("status") for r in recs)
    print(f"\n{len(recs)} records processed: {dict(status_counts)}")

    failed = [(r.get("error") or "Unknown error")[:160] for r in recs if r.get("status") != "ok"]
    if failed:
        print("\nTop errors:")
        for err, count in collections.Counter(failed).most_common(5):
            print(f"  x{count}  {err}")

Data directory:  True
Manifest exists: True
Hyp RTTM count:  173

100 records processed: {'ok': 99, 'failed': 1}

Top errors:
  x1  Unknown error


In [13]:
# import json, collections, pathlib

# d = pathlib.Path('/kaggle/working/data/hyp/sortformer')
# print('dir exists:', d.exists())
# if d.exists():
#     print('contents:', sorted(p.name for p in d.iterdir()))
#     print('rttms   :', len(list((d / 'rttm').glob('*.rttm'))) if (d / 'rttm').exists() else 0)

# p = d / 'manifest.jsonl'
# if not p.exists():
#     print('NO MANIFEST -- the sortformer diarize run never started')
# else:
#     recs = [json.loads(l) for l in p.open(encoding='utf-8') if l.strip()]
#     print(len(recs), 'records', collections.Counter(r['status'] for r in recs))
#     errs = collections.Counter((r.get('error') or '')[:200]
#                                for r in recs if r['status'] != 'ok')
#     for e, n in errs.most_common(5):
#         print(f'  x{n}  {e}')

In [14]:
!pip install -q "nemo_toolkit[asr]"

In [15]:
import torch
from nemo.collections.asr.models import SortformerEncLabelModel

MODEL_ID = "nvidia/diar_sortformer_4spk-v1"

m = SortformerEncLabelModel.from_pretrained(MODEL_ID)
m.eval()
m.to(torch.device("cuda"))

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


diar_sortformer_4spk-v1.nemo:   0%|          | 0.00/493M [00:00<?, ?B/s]

[NeMo W 2026-09-07 17:31:50 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-09-07 17:31:50 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-09-07 17:31:53 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.


SortformerEncLabelModel(
  (preprocessor): AudioToMelSpectrogramPreprocessor(
    (featurizer): FilterbankFeatures()
  )
  (encoder): ConformerEncoder(
    (pre_encode): ConvSubsampling(
      (out): Linear(in_features=2560, out_features=512, bias=True)
      (conv): MaskedConvSequential(
        (0): Conv2d(1, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        (3): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
        (4): ReLU(inplace=True)
        (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        (6): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
        (7): ReLU(inplace=True)
      )
    )
    (pos_enc): RelPositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-17): 18 x ConformerLayer(
        (norm_feed_forward1): LayerNorm((512,), eps=1

In [16]:
# wav = sorted(AUDIO.glob("*.wav"))[0]
# pred = m.diarize(audio=[str(wav)], batch_size=1)

# print("type :", type(pred))
# print("len  :", len(pred))
# inner = pred[0]
# print("inner:", type(inner), len(inner) if hasattr(inner, "__len__") else "-")

# items = inner if isinstance(inner, list) else pred
# for x in items[:3]:
#     print("  ", type(x).__name__, repr(x)[:200])

In [17]:
!python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO} --limit 5

[env ] system=sortformer  model=nvidia/diar_sortformer_4spk-v1
[env ] wav_dir=/kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
[env ] device=cuda  gpu=Tesla T4
[warn] sortformer is capped at 4 speakers; clips above that cannot be solved and should be reported separately
[run ] 2 of 5 clips to do (3 already done)

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-09-07 17:32:06 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_siz

In [18]:
# !python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO}

In [19]:
# import inspect
# print(inspect.signature(m.diarize))
# print("---")
# sm = getattr(m, "sortformer_modules", None)
# print([a for a in dir(sm) if not a.startswith("_")] if sm else "no sortformer_modules")
# print("---")
# print((m.diarize.__doc__ or "")[:1500])

In [20]:
# from omegaconf import OmegaConf
# import dataclasses
# from nemo.collections.asr.parts.mixins.diarization import DiarizeConfig

# print("model streaming attrs:", [a for a in dir(m) if "stream" in a.lower()])
# print("model forward attrs  :", [a for a in dir(m) if "forward" in a.lower()])
# print("---DiarizeConfig fields---")
# print([f.name for f in dataclasses.fields(DiarizeConfig)])
# print("---cfg keys---")
# print(list(m.cfg.keys()))
# for k in m.cfg:
#     if "stream" in k.lower() or "chunk" in k.lower():
#         print(k, "=", OmegaConf.to_container(m.cfg[k]) if hasattr(m.cfg[k], "keys") else m.cfg[k])


In [21]:
import pathlib, subprocess
CODE = pathlib.Path("/kaggle/input/datasets/ritankarmondal/sarvam-diar-code")   # adjust if your slug differs
src = CODE / "upload_code" / "stage3_diarize.py"
print("source exists:", src.exists())
if src.exists():
    txt = src.read_text(encoding="utf-8")
    print("source has sortformer_stream:", txt.count("sortformer_stream"))
    print("source size:", src.stat().st_size)   # new file is 18463 bytes

source exists: True
source has sortformer_stream: 6
source size: 18463


In [22]:
!cp /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code/*.py /kaggle/working/
!grep -c sortformer_stream /kaggle/working/stage3_diarize.py

6


In [23]:
!grep -c sortformer_stream /kaggle/working/stage3_diarize.py

6


In [24]:
!python stage3_diarize.py --system sortformer_stream --data data --wav-dir {AUDIO} --limit 5

[env ] system=sortformer_stream  model=nvidia/diar_sortformer_4spk-v1
[env ] wav_dir=/kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
[env ] device=cuda  gpu=Tesla T4
[warn] sortformer_stream is capped at 4 speakers; clips above that cannot be solved and should be reported separately
[run ] 5 of 5 clips to do (0 already done)

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-09-07 17:32:51 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null

In [25]:
!python stage3_diarize.py --system sortformer_stream --data data --wav-dir {AUDIO}

[env ] system=sortformer_stream  model=nvidia/diar_sortformer_4spk-v1
[env ] wav_dir=/kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
[env ] device=cuda  gpu=Tesla T4
[warn] sortformer_stream is capped at 4 speakers; clips above that cannot be solved and should be reported separately
[run ] 94 of 99 clips to do (5 already done)

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-09-07 17:33:40 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: nu